# French Editorial Fingerprinting — Outlet Classification from **Headlines Only**

**Question.** Within a single broad news domain (International / Politique / Économie),
does outlet-specific editorial *style* remain identifiable from article text — even after
explicit outlet mentions are masked?

This is an **outlet** classification task (predict the publisher), not a topic task. Topic is
held roughly constant by restricting to three hard-news categories and balancing every
`outlet × category` cell.

**Design (pre-specified in the task brief).**
- 10 French outlets, 3 categories, 2021–2025.
- Balanced sampling: target 1,500 articles / `outlet × category` cell (fallback 1,000; else largest safe common cap).
- Two split strategies: random stratified (80/10/10) and **temporal** (train 2021–2024, val late-2024, test 2025).
- Models: majority baseline, word TF-IDF + LogReg, char TF-IDF + LinearSVM, CamemBERT.
- - Masked-outlet is the key control: above-chance accuracy there is evidence of genuine stylistic fingerprinting.

**Data source.** Headlines are reconstructed from `data/corpus_manifest.csv` (article ids, hashes, labels, and public URLs; see README A). This notebook consumes the rebuilt pool and performs cleaning, balanced sampling, splitting, and model training. No database or cloud storage is required.


## Environment setup
Run once per Colab session. Skip locally if deps are already present.

In [ ]:
# @title
# Colab dependency install (idempotent). Comment out if running locally with deps present.
import sys, subprocess, importlib
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Core stack. Transformers/torch are usually preinstalled on Colab GPU runtimes.
_pip("pyarrow", "scikit-learn>=1.3", "pandas", "matplotlib", "spacy>=3.7", "datasets")

# French NER model for entity masking — install it, don't hope it's there.
if importlib.util.find_spec("fr_core_news_lg") is None:
    subprocess.run([sys.executable, "-m", "spacy", "download", "fr_core_news_lg"], check=False)

# Uncomment on a fresh runtime if needed:
# _pip("transformers>=4.40", "accelerate>=0.30", "torch")
print("setup done")

## 1. Config
All knobs in one place. Edit here, rerun top-to-bottom.

In [ ]:
import os, re, json, math, random, hashlib, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Panel: 10 French national outlets (outlet_id -> display name).
#      Selection rationale + ACPM anchoring: see the paper's Data section.
ALL_NAMES = {
    1: "Le Monde", 3: "Le Figaro", 10: "Les Echos", 25: "Libération", 14: "La Croix",
    19: "Le Parisien", 26: "Franceinfo", 27: "BFMTV", 21: "TF1 INFO", 17: "Ouest-France",
}
PANELS = {"n10": [1, 3, 10, 25, 14, 19, 26, 27, 21, 17]}
PANEL_TAG = "n10"
OUTLETS = {i: ALL_NAMES[i] for i in PANELS[PANEL_TAG]}
OUTLET_DOMAINS = {  # outlet-mention masking patterns
    "Le Monde": ["lemonde", "le monde"], "Le Figaro": ["lefigaro", "le figaro", "figaro"],
    "Les Echos": ["lesechos", "les echos", "les echos"], "La Croix": ["la-croix", "la croix"],
    "Ouest-France": ["ouest-france", "ouest france"], "Le Parisien": ["leparisien", "le parisien"],
    "TF1 INFO": ["tf1info", "tf1 info", "tf1", "lci"], "Libération": ["liberation", "libération", "libe", "libé"],
    "Franceinfo": ["franceinfo", "france info", "francetvinfo"], "BFMTV": ["bfmtv", "bfm tv", "bfm"],
}
CATEGORIES = ["International", "Politique", "Économie"]
CAT_PREFIX = {
    "International": "[CAT=INTERNATIONAL]", "Politique": "[CAT=POLITIQUE]", "Économie": "[CAT=ECONOMIE]",
}

# ---- Date window -----------------------------------------------------
DATE_START = "2018-01-01"
DATE_END_EXCL = "2026-01-01"
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

# ---- Cleaning thresholds --------------------------------------------
MIN_HEADLINE_CHARS = 10
MIN_HEADLINE_TOKENS = 2

# ---- Balanced sampling ----------------------------------------------
TARGET_PER_CELL = 5672          # 17,016/outlet; 56,720/category (choose_cap auto-lowers if a cell is short)
FALLBACK_PER_CELL = 2000
POOL_PER_CELL = 10000           # oversample per cell for dedup / short-drop / temporal headroom

# ---- Temporal split --------------------------------------------------
TEMPORAL_TRAIN_YEARS = [2020, 2021, 2022, 2023, 2024]
TEMPORAL_TEST_YEARS = [2025]
TEMPORAL_VAL_FRACTION = 0.10

# ---- Models / runtime toggles ---------------------------------------
RUN_CAMEMBERT = True
CAMEMBERT_MODEL = "camembert/camembert-large"  # 24L/1024H (~336M params)
CAMEMBERT_MAXLEN = 64
CAMEMBERT_BATCH = 256           # drop to 64 if OOM
CAMEMBERT_EPOCHS = 10           # early stopping (patience=2) caps it
CAMEMBERT_LR = 1e-5             # locked best LR (prior sweep over {5,2,1}e-5)
CAMEMBERT_WD = 0.01
CAMEMBERT_WARMUP_RATIO = 0.1
CAMEMBERT_LR_SWEEP = [1e-5]
CAMEMBERT_VARIANTS = [("headline_only", "raw"), ("headline_only", "mask_outlet")]

print("Config loaded. Outlets:", len(OUTLETS), "| Categories:", CATEGORIES)


## 2. Load the rebuilt corpus

Train on the corpus rebuilt from `data/corpus_manifest.csv` (README A). Point `ART` at a local output directory and load the article pool; the cells below then clean, balance, split, and train exactly as in the paper.


In [ ]:
# Local outputs (the authors' run wrote these to cloud storage; reviewers use a local dir).
ART = Path("results") / PANEL_TAG
ART.mkdir(parents=True, exist_ok=True)

# The article pool rebuilt from data/corpus_manifest.csv (README A):
#   columns: article_id, outlet, category, date, title, text, url, year
POOL_PATH = Path("results") / "articles_pool.parquet"
df_pool = pd.read_parquet(POOL_PATH)
print("Loaded pool:", len(df_pool), "rows | outlets =", df_pool["outlet"].nunique())


## 3. Clean and normalize
Whitespace normalization, drop missing outlet/text, dedup, drop very-short articles.

In [ ]:
# @title
import html, unicodedata
def normalize_ws(s):
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)                                              # decode &#8217; &amp; etc.
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C") # strip control/format chars (LRM/RLM/ZWSP)
    s = re.sub(r"\blrm\b", " ", s)                                    # residual literal left-to-right-mark token
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# --- Aggressive extraction-artifact cleaning (authoritative run) ----------------------
# Audit findings: some outlets' <title> carried a templated site-name suffix (e.g. " - L'Humanite"),
# which leaked outlet identity past mask_outlet; broadcast/format tags ("VIDEO -", "EN DIRECT :") and
# rare section/index landing pages also contaminate the headline string. We strip these from the
# as-published headline BEFORE building the raw/mask variants, so no model can exploit them.
_OUTLET_SUFFIX_PAT = re.compile(
    r"\s*[-|\u2013\u2014]\s*(" + "|".join(sorted(
        (re.escape(v) for variants in OUTLET_DOMAINS.values() for v in variants),
        key=len, reverse=True)) + r")\s*$", re.I)
_LEAD_TAG_PAT = re.compile(
    r"^\s*(EN\s+DIRECT|DIRECT|VID[E\u00c9]O|REPLAY|LIVE|PHOTOS?|EN\s+IMAGES|INFOGRAPHIES?|CARTE|"
    r"PODCAST|[E\u00c9]DITO|TRIBUNE|D[E\u00c9]CRYPTAGE|REPORTAGE|T[E\u00c9]MOIGNAGE|ENTRETIEN|INTERVIEW)"
    r"\s*[:.\-\u2013\u2014]\s+", re.I)
_INDEX_PAT = re.compile(
    r"(?i)(toute l.actualit|actualit[e\u00e9]s?\s*,\s*infos|donn[e\u00e9]es pays|infos? et actualit|"
    r"tous les articles|derni[e\u00e8]res actualit|fil info|toute l.info)")

def clean_headline(s):
    s = normalize_ws(s)
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)            # stray URLs / domains
    prev = None
    while prev != s:                                          # strip repeated trailing site/section suffixes
        prev = s
        s = _OUTLET_SUFFIX_PAT.sub("", s)
    s = _LEAD_TAG_PAT.sub("", s)                              # strip a leading broadcast/format tag
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"^[\-|\u2013\u2014:.\s]+|[\s\-|\u2013\u2014]+$", "", s).strip()  # stray edge separators
    return s

def is_index_page(s):
    return bool(_INDEX_PAT.search(str(s)))

def clean_pool(df):
    df = df.copy()
    n0 = len(df)
    df["title"] = df["title"].map(clean_headline)   # aggressive artifact cleaning (see above)
    df["text"] = df["text"].map(normalize_ws)
    # Drop missing outlet / empty-after-clean title.
    df = df[df["outlet"].notna() & df["title"].astype(bool)]
    # Drop section/index landing pages (not real articles).
    df = df[~df["title"].map(is_index_page)]
    n_missing = n0 - len(df)
    # Parse date; keep only rows with a usable year in range.
    df["dt"] = pd.to_datetime(df["date"], errors="coerce", utc=False)
    df = df[df["dt"].notna()]
    df["year"] = df["dt"].dt.year
    df = df[df["year"].isin(YEARS)]
    # Drop very-short articles.
    df = df[(df["title"].str.len() >= MIN_HEADLINE_CHARS) &
            (df["title"].str.split().map(len) >= MIN_HEADLINE_TOKENS)]
    n_short = n0 - n_missing - len(df)
    # Deduplicate: exact body hash, and (outlet,title) near-dupes.
    df["_hash"] = df["title"].map(lambda t: hashlib.md5(t.encode("utf-8")).hexdigest())
    df = df.drop_duplicates(subset="_hash")
    df = df.drop_duplicates(subset=["outlet", "title"])
    df = df.drop(columns=["_hash"])
    print(f"clean: {n0:,} -> {len(df):,}  (missing={n_missing:,}, short/oob removed, deduped)")
    return df.reset_index(drop=True)

df_clean = clean_pool(df_pool)
df_clean.head(2)


## 4. Filter categories and date range
Enforce the category whitelist and date window (defensive — extraction already filtered).

In [ ]:
# @title
df = df_clean[df_clean["category"].isin(CATEGORIES)].copy()
df = df[(df["date"] >= DATE_START) & (df["date"] < DATE_END_EXCL)]
df = df[df["outlet"].isin(list(OUTLETS.values()))]   # restrict to the active modeling panel
df["outlet"] = pd.Categorical(df["outlet"], categories=list(OUTLETS.values()))
df["category"] = pd.Categorical(df["category"], categories=CATEGORIES)
print("After filter:", df.shape)
print(df[["outlet", "category", "year"]].describe(include="all"))

## 5. Compute outlet / category counts
Transparent availability before sampling. Saves `dataset_summary.csv`.

In [ ]:
# @title
def summary_tables(df):
    by_outlet = df["outlet"].value_counts().rename_axis("outlet").reset_index(name="count")
    by_cat = df["category"].value_counts().rename_axis("category").reset_index(name="count")
    cell = (df.groupby(["outlet", "category"], observed=True).size()
              .reset_index(name="count"))
    pivot = cell.pivot(index="outlet", columns="category", values="count").fillna(0).astype(int)
    return by_outlet, by_cat, cell, pivot

by_outlet, by_cat, cell_counts, cell_pivot = summary_tables(df)
print("Available per outlet:\n", by_outlet.to_string(index=False))
print("\nAvailable per category:\n", by_cat.to_string(index=False))
print("\nAvailable per outlet x category cell:\n", cell_pivot)

dataset_summary = cell_counts.copy()
dataset_summary.to_csv(ART / "dataset_summary.csv", index=False)

## 6. Balanced sampling

Pick a per-cell cap: prefer **1,500**, fall back to **1,000**, else the **largest safe common cap**
(the smallest cell count, if even 1,000 is infeasible). Then sample exactly that many per
`outlet × category` cell. Deterministic (seeded). Saves `sampling_summary.csv`.

In [ ]:
# @title
def choose_cap(cell_pivot):
    min_cell = int(cell_pivot.values.min())
    if min_cell >= TARGET_PER_CELL:
        return TARGET_PER_CELL, "target"
    return min_cell, "common_cap"   # largest balanced size the data supports; no round-down waste

CAP, cap_reason = choose_cap(cell_pivot)
print(f"Per-cell cap = {CAP} ({cap_reason}); smallest available cell = {int(cell_pivot.values.min())}")

def balanced_sample(df, cap):
    parts = []
    for (o, c), g in df.groupby(["outlet", "category"], observed=True):
        n = min(cap, len(g))
        parts.append(g.sample(n=n, random_state=SEED))
    out = pd.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

df_sample = balanced_sample(df, CAP)

samp_cell = (df_sample.groupby(["outlet", "category"], observed=True).size()
             .reset_index(name="sampled"))
sampling_summary = samp_cell.merge(cell_counts.rename(columns={"count": "available"}),
                                   on=["outlet", "category"], how="left")
sampling_summary["cap"] = CAP
sampling_summary["cap_reason"] = cap_reason
sampling_summary.to_csv(ART / "sampling_summary.csv", index=False)

print("Final sampled total:", len(df_sample))
print("\nSampled per outlet x category:\n",
      samp_cell.pivot(index="outlet", columns="category", values="sampled").fillna(0).astype(int))
print("\nSampled per outlet:\n", df_sample["outlet"].value_counts().to_string())
print("\nSampled per category:\n", df_sample["category"].value_counts().to_string())

## 7. Build text variants

**Fields:** `headline_only` only (article headlines / titles). Preprocessing variants
(raw, masked-outlet, masked-entities) are applied to headlines identically.
**Preprocessing:** `raw`, `mask_outlet`, `mask_entities`. `mask_outlet` replaces outlet
names/domains/URLs with `[OUTLET]`/`[URL]`. `mask_entities` additionally replaces spaCy
`fr_core_news_lg` PER/LOC/ORG/MISC spans with `[ENT]` (the model is installed in the setup cell).

In [ ]:
# @title
# --- Outlet-mention masking patterns (built from OUTLET_DOMAINS) -----
_outlet_patterns = []
for variants in OUTLET_DOMAINS.values():
    for v in variants:
        _outlet_patterns.append(re.escape(v))
OUTLET_RE = re.compile(r"(?i)\b(" + "|".join(sorted(_outlet_patterns, key=len, reverse=True)) + r")\b")
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def mask_outlet(text):
    text = URL_RE.sub("[URL]", text)
    return OUTLET_RE.sub("[OUTLET]", text)

# --- NER masking: spaCy fr_core_news_lg (installed in the setup cell) ----
import spacy
# Keep only tok2vec + ner for speed; the model is required, not optional.
_NLP = spacy.load("fr_core_news_lg",
                  disable=["morphologizer", "parser", "attribute_ruler", "lemmatizer"])

def mask_entities(text):
    text = mask_outlet(text)
    doc = _NLP(text[:100000])
    spans = [(e.start_char, e.end_char) for e in doc.ents
             if e.label_ in {"PER", "LOC", "ORG", "MISC"}]
    for s, e in sorted(spans, reverse=True):
        text = text[:s] + "[ENT]" + text[e:]
    return text

# --- Field builders --------------------------------------------------
def field_headline(r):
    return r["title"]

FIELD_FN = {"headline_only": field_headline}
PREPROC_FN = {"raw": lambda s: s, "mask_outlet": mask_outlet, "mask_entities": mask_entities}

def build_text(df, text_field, preprocess, cat_prefix=False):
    base = df.apply(FIELD_FN[text_field], axis=1)
    fn = PREPROC_FN[preprocess]
    out = base.map(fn)
    if cat_prefix:
        out = df["category"].astype(str).map(CAT_PREFIX) + " " + out
    return out

# Smoke test on one row.
_demo = df_sample.iloc[0]
print("HEADLINE:", build_text(df_sample.iloc[:1], "headline_only", "raw").iloc[0][:160], "...")
print("MASKED: ", build_text(df_sample.iloc[:1], "headline_only", "mask_outlet").iloc[0][:160], "...")

## 8. Split A — random stratified (80 / 10 / 10), stratified by `outlet × category`.

In [ ]:
# @title
from sklearn.model_selection import train_test_split

def stratified_split(df):
    strat = df["outlet"].astype(str) + "|" + df["category"].astype(str)
    tr, tmp = train_test_split(df, test_size=0.20, random_state=SEED, stratify=strat)
    strat_tmp = tmp["outlet"].astype(str) + "|" + tmp["category"].astype(str)
    val, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=strat_tmp)
    return (tr.reset_index(drop=True), val.reset_index(drop=True), te.reset_index(drop=True))

rand_train, rand_val, rand_test = stratified_split(df_sample)
print("Random split:", len(rand_train), len(rand_val), len(rand_test))

## 9. Split B — temporal (mandatory)

Train = 2021–2024, **Test = 2025 only**, Validation = held-out late-2024 slice carved from train.
This measures generalization to *future* articles — the realistic deployment setting and a
stronger test of stylistic (vs topical/temporal) signal.

In [ ]:
# @title
def temporal_split(df):
    train_pool = df[df["year"].isin(TEMPORAL_TRAIN_YEARS)].copy()
    test = df[df["year"].isin(TEMPORAL_TEST_YEARS)].copy()
    # Validation: latest TEMPORAL_VAL_FRACTION of 2024 by date, stratified-ish by cell.
    y2024 = train_pool[train_pool["year"] == 2024].sort_values("dt")
    n_val = int(len(y2024) * TEMPORAL_VAL_FRACTION)
    val = y2024.tail(n_val)
    train = train_pool.drop(index=val.index)
    return (train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True))

temp_train, temp_val, temp_test = temporal_split(df_sample)
print("Temporal split:", len(temp_train), len(temp_val), len(temp_test))
print("Test years:", sorted(temp_test["year"].unique()),
      "| Train years:", sorted(temp_train["year"].unique()))
print("Per-cell test counts:\n",
      temp_test.groupby(["outlet","category"], observed=True).size()
      .unstack(fill_value=0))

SPLITS = {
    "random":   (rand_train, rand_val, rand_test),
    "temporal": (temp_train, temp_val, temp_test),
}

## Evaluation helpers
Shared metric / prediction utilities used by every model below.

In [ ]:
# @title
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
LABELS = list(OUTLETS.values())

def top2_accuracy(y_true, proba, labels):
    if proba is None:
        return np.nan
    idx = np.argsort(proba, axis=1)[:, -2:]
    top2 = {i: set(labels[j] for j in row) for i, row in enumerate(idx)}
    return np.mean([yt in top2[i] for i, yt in enumerate(y_true)])

def eval_metrics(y_true, y_pred, proba=None, labels=LABELS):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0),
        "top2_acc": top2_accuracy(list(y_true), proba, labels),
        "n": len(y_true),
    }

def per_class_df(y_true, y_pred, labels=LABELS):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return pd.DataFrame({"outlet": labels, "precision": p, "recall": r, "f1": f, "support": s})

RESULTS = []          # accumulates one metrics row per (experiment, split)
PER_CLASS = []        # accumulates per-class tables
PREDICTIONS = []      # accumulates sample predictions
CONF_MATRICES = {}    # (best models) split -> confusion DataFrame
FEATURE_IMP = []      # linear-model top features

In [ ]:
# @title
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

def subset_for_family(df, family, category):
    return df if family == "pooled" else df[df["category"] == category]

def make_vectorizer(kind):
    if kind == "word_lr":
        return TfidfVectorizer(ngram_range=(1, 2), max_features=120000,
                               sublinear_tf=True, min_df=2)
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                           max_features=200000, sublinear_tf=True, min_df=2)

def make_clf(kind):
    if kind == "word_lr":
        return LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced",
                                  n_jobs=-1, multi_class="auto")
    return LinearSVC(C=1.0, class_weight="balanced")

def run_linear(kind, family, category, text_field, preprocess, split_name,
               cat_prefix=False, collect_conf=False, collect_feat=False):
    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return
    Xtr = build_text(tr, text_field, preprocess, cat_prefix)
    Xte = build_text(te, text_field, preprocess, cat_prefix)
    ytr = tr["outlet"].astype(str); yte = te["outlet"].astype(str).tolist()
    labels = sorted(ytr.unique().tolist())

    vec = make_vectorizer(kind); base = make_clf(kind)
    Xtr_v = vec.fit_transform(Xtr); Xte_v = vec.transform(Xte)
    proba = None
    if kind == "word_lr":
        base.fit(Xtr_v, ytr)
        y_pred = base.predict(Xte_v)
        proba = base.predict_proba(Xte_v); proba_labels = list(base.classes_)
    else:
        # LinearSVC has no probabilities; calibrate for top-2 only when cheap.
        base.fit(Xtr_v, ytr)
        y_pred = base.predict(Xte_v)
        proba_labels = labels
    # top-2 needs proba aligned to LABELS; build aligned matrix if available.
    proba_full = None
    if proba is not None:
        proba_full = np.zeros((len(yte), len(LABELS)))
        for j, lab in enumerate(proba_labels):
            if lab in LABELS:
                proba_full[:, LABELS.index(lab)] = proba[:, j]

    m = eval_metrics(yte, y_pred, proba_full)
    name = f"{kind}|{family}|{category}|{text_field}|{preprocess}{'|catpfx' if cat_prefix else ''}"
    m.update({"experiment": name, "model": kind, "family": family, "category": category,
              "text_field": text_field, "preprocess": preprocess, "split": split_name})
    RESULTS.append(m)
    pc = per_class_df(yte, y_pred, labels); pc["experiment"] = name; pc["split"] = split_name
    PER_CLASS.append(pc)
    print(f"[{name}/{split_name}] acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f} "
          f"top2={m['top2_acc'] if not np.isnan(m['top2_acc']) else float('nan'):.3f}")

    if collect_conf:
        cm = confusion_matrix(yte, y_pred, labels=LABELS)
        CONF_MATRICES[split_name] = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    if collect_feat and kind == "word_lr":
        extract_word_features(vec, base, name)
    if collect_feat and kind == "char_svm":
        extract_char_features(vec, base, name)
    # stash predictions for error analysis (only the flagged best run)
    if collect_conf:
        conf = proba_full.max(axis=1) if proba_full is not None else np.full(len(yte), np.nan)
        dfp = te[["article_id", "outlet", "category", "year"]].copy()
        dfp["text_len"] = build_text(te, text_field, preprocess, cat_prefix).str.len().values
        dfp["pred"] = y_pred; dfp["confidence"] = conf
        dfp["correct"] = (dfp["outlet"].astype(str) == dfp["pred"])
        dfp["experiment"] = name; dfp["split"] = split_name
        PREDICTIONS.append(dfp)
    return m

def extract_word_features(vec, clf, name, topn=20):
    feats = np.array(vec.get_feature_names_out())
    for i, lab in enumerate(clf.classes_):
        coef = clf.coef_[i] if clf.coef_.shape[0] > 1 else clf.coef_[0]
        top = np.argsort(coef)[-topn:][::-1]
        for rank, j in enumerate(top):
            FEATURE_IMP.append({"experiment": name, "model": "word_lr", "outlet": lab,
                                "rank": rank + 1, "feature": feats[j], "weight": float(coef[j])})

def extract_char_features(vec, clf, name, topn=20):
    feats = np.array(vec.get_feature_names_out())
    classes = clf.classes_
    for i, lab in enumerate(classes):
        coef = clf.coef_[i] if clf.coef_.shape[0] > 1 else clf.coef_[0]
        top = np.argsort(coef)[-topn:][::-1]
        for rank, j in enumerate(top):
            FEATURE_IMP.append({"experiment": name, "model": "char_svm", "outlet": lab,
                                "rank": rank + 1, "feature": repr(feats[j]), "weight": float(coef[j])})

In [ ]:
# @title
# ---- CharCNN experiments: pure-char (A) and char+word hybrid (B) --------
# Upgrade from the previous 4-layer stack to a deeper 6-layer architecture
# (Zhang et al. style), compare pure-char vs char+word to isolate which branch
# carries the signal, and tune training (lr 5e-4, patience 3, 30 epochs).

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- Vocab builders (train-only) ----------------------------------------
def build_char_vocab(texts):
    chars = set()
    for t in texts:
        chars.update(t)
    return {c: i+1 for i, c in enumerate(sorted(chars))}

def build_word_vocab(texts, min_freq=2):
    from collections import Counter
    c = Counter()
    for t in texts:
        c.update(t.split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, f in c.most_common():
        if f >= min_freq:
            vocab[w] = len(vocab)
    return vocab

# --- Encoding helpers ----------------------------------------------------
def text_to_char_ids(text, char2idx, max_len=200):
    ids = [char2idx.get(c, 0) for c in text[:max_len]]
    return ids + [0] * (max_len - len(ids))

def text_to_word_ids(text, word2idx, max_len=50):
    words = text.split()[:max_len]
    ids = [word2idx.get(w, 1) for w in words]
    return ids + [0] * (max_len - len(ids))

# --- Dataset -------------------------------------------------------------
class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, char2idx, word2idx):
        self.texts = texts
        self.labels = labels
        self.char2idx = char2idx
        self.word2idx = word2idx
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        c = torch.tensor(text_to_char_ids(self.texts[i], self.char2idx), dtype=torch.long)
        w = torch.tensor(text_to_word_ids(self.texts[i], self.word2idx), dtype=torch.long)
        return c, w, torch.tensor(self.labels[i], dtype=torch.long)

# --- Shared 6-layer char conv tower (Zhang et al.) ----------------------
# 4 max-pool layers of stride 3 on 200-char input: 200->66->22->7->2
# then adaptive_max_pool1d(1) collapses to a 256-dim char vector.
def _build_char_tower(char_emb):
    return nn.Sequential(
        nn.Conv1d(char_emb, 64,  3, padding=1), nn.ReLU(), nn.MaxPool1d(3),
        nn.Conv1d(64,       64,  3, padding=1), nn.ReLU(), nn.MaxPool1d(3),
        nn.Conv1d(64,      128,  3, padding=1), nn.ReLU(), nn.MaxPool1d(3),
        nn.Conv1d(128,     128,  3, padding=1), nn.ReLU(), nn.MaxPool1d(3),
        nn.Conv1d(128,     256,  1),             nn.ReLU(),
        nn.Conv1d(256,     256,  1),             nn.ReLU(),
    )

# --- Version A: pure CharCNN (no word branch) ----------------------------
class CharCNNPureModel(nn.Module):
    def __init__(self, num_chars, num_classes, char_emb=16, dropout=0.5):
        super().__init__()
        self.char_embed = nn.Embedding(num_chars + 1, char_emb, padding_idx=0)
        self.char_tower = _build_char_tower(char_emb)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, char_ids, _word_ids=None):
        x = self.char_embed(char_ids).permute(0, 2, 1)
        x = self.char_tower(x)
        x = F.adaptive_max_pool1d(x, 1).squeeze(-1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

# --- Version B: CharCNN + word-embedding mean hybrid --------------------
class CharCNNWordModel(nn.Module):
    def __init__(self, num_chars, num_words, num_classes,
                 char_emb=16, word_emb=64, dropout=0.5):
        super().__init__()
        self.char_embed = nn.Embedding(num_chars + 1, char_emb, padding_idx=0)
        self.char_tower = _build_char_tower(char_emb)
        self.word_embed = nn.Embedding(num_words, word_emb, padding_idx=0)
        self.fc1 = nn.Linear(256 + word_emb, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, char_ids, word_ids):
        x = self.char_embed(char_ids).permute(0, 2, 1)
        x = self.char_tower(x)
        x = F.adaptive_max_pool1d(x, 1).squeeze(-1)
        w = self.dropout(self.word_embed(word_ids).mean(dim=1))
        out = torch.cat([x, w], dim=1)
        out = self.dropout(F.relu(self.fc1(out)))
        return self.fc2(out)

# --- Training & eval -----------------------------------------------------
def run_char_cnn(family, category, text_field, preprocess, split_name,
                 model_type='pure', device='cuda'):
    # model_type: 'pure' (char only) or 'word' (char + word mean)
    from sklearn.metrics import f1_score, accuracy_score

    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category)
    va = subset_for_family(val, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return

    Xtr = build_text(tr, text_field, preprocess).tolist()
    Xva = build_text(va, text_field, preprocess).tolist()
    Xte = build_text(te, text_field, preprocess).tolist()
    ytr = tr['outlet'].astype(str).tolist()
    yva = va['outlet'].astype(str).tolist()
    yte = te['outlet'].astype(str).tolist()

    labels = sorted(list(set(ytr)))
    lab2id = {l: i for i, l in enumerate(labels)}
    ytr_i = [lab2id[l] for l in ytr]
    yva_i = [lab2id[l] for l in yva]
    yte_i = [lab2id[l] for l in yte]

    char2idx = build_char_vocab(Xtr)
    word2idx = build_word_vocab(Xtr)

    train_ds = HeadlineDataset(Xtr, ytr_i, char2idx, word2idx)
    val_ds   = HeadlineDataset(Xva, yva_i, char2idx, word2idx)
    test_ds  = HeadlineDataset(Xte, yte_i, char2idx, word2idx)

    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=64)
    test_dl  = DataLoader(test_ds,  batch_size=64)

    if model_type == 'pure':
        model = CharCNNPureModel(len(char2idx), len(labels)).to(device)
        model_key = 'charcnn_pure'
    else:
        model = CharCNNWordModel(len(char2idx), len(word2idx), len(labels)).to(device)
        model_key = 'charcnn_word'

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

    best_f1, best_state, patience_counter = -1, None, 0
    PATIENCE, MAX_EPOCHS = 3, 30

    name = f'{model_key}|{family}|{category}|{text_field}|{preprocess}'
    print(f"[{name}/{split_name}] {model_type} arch | {len(train_ds)} train examples")
    for epoch in range(MAX_EPOCHS):
        model.train()
        for c, w, y in train_dl:
            c, w, y = c.to(device), w.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(c, w), y)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, truths = [], []
        with torch.no_grad():
            for c, w, y in val_dl:
                preds.extend(model(c.to(device), w.to(device)).argmax(1).cpu().numpy())
                truths.extend(y.numpy())
        val_f1 = f1_score(truths, preds, average='macro', zero_division=0)
        val_acc = accuracy_score(truths, preds)
        print(f"  Epoch {epoch+1:2d} -- val_macro_f1={val_f1:.4f}  val_acc={val_acc:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (patience={PATIENCE})")
                break

    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for c, w, _ in test_dl:
            out = model(c.to(device), w.to(device))
            all_probs.extend(F.softmax(out, dim=1).cpu().numpy())
            all_preds.extend(out.argmax(1).cpu().numpy())

    proba = np.array(all_probs)
    proba_full = np.zeros((len(yte), len(LABELS)))
    for j, lab in enumerate(labels):
        if lab in LABELS:
            proba_full[:, LABELS.index(lab)] = proba[:, j]

    y_pred_str = [labels[i] for i in all_preds]
    m = eval_metrics(yte, y_pred_str, proba_full)
    m.update({'experiment': name, 'model': model_key, 'family': family,
              'category': category, 'text_field': text_field,
              'preprocess': preprocess, 'split': split_name})
    RESULTS.append(m)
    pc = per_class_df(yte, y_pred_str, LABELS)
    pc['experiment'] = name; pc['split'] = split_name
    PER_CLASS.append(pc)
    print(f"  >>> TEST [{name}/{split_name}]  acc={m['accuracy']:.3f}  "
          f"macroF1={m['macro_f1']:.3f}  top2={m['top2_acc']:.3f}  "
          f"(best_val_f1={best_f1:.4f})")


In [ ]:
# @title
# --- CamemBERT fine-tuning (improved) -----------------------------------
# Key changes vs baseline:
#   - LR sweep on raw/random to pick best LR before running all variants
#   - warmup_ratio=0.1  (prevents early destructive gradient updates)
#   - early_stopping_patience 1->2  (less hair-trigger)
#   - label_smoothing_factor=0.1  (regularises 10-class softmax)
#   - max_len 256->64  (headlines are short; speeds up training ~4x)
#   - batch 16->32  (fits on GPU with shorter seqs, better gradient estimates)
#   - epochs 3->10 with early stopping  (more headroom to converge)

# --- Load tokenizer and base model ONCE globally ----------------------
_tok = None
_base_model = None

def _load_camembert_once():
    global _tok, _base_model
    if _tok is not None:
        return _tok, _base_model
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    _tok = AutoTokenizer.from_pretrained(CAMEMBERT_MODEL)
    _base_model = AutoModelForSequenceClassification.from_pretrained(
        CAMEMBERT_MODEL, num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id={l: i for i, l in enumerate(LABELS)},
    )
    if torch.cuda.is_available():
        _base_model = _base_model.cuda()
    return _tok, _base_model

from datasets import Dataset
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback

def tokenize_fn(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True, max_length=CAMEMBERT_MAXLEN)

def _make_dataset(frame, tok, text_field, preprocess):
    texts = build_text(frame, text_field, preprocess).tolist()
    labs = [LABELS.index(o) for o in frame["outlet"].astype(str)]
    ds = Dataset.from_dict({"text": texts, "labels": labs})
    ds = ds.map(lambda ex: tokenize_fn(ex, tok), batched=True, remove_columns=["text"])
    return ds

def run_camembert(family, category, text_field, preprocess, split_name, lr=None, collect_conf=False):
    import torch, copy
    from sklearn.metrics import f1_score, accuracy_score

    lr = lr if lr is not None else CAMEMBERT_LR

    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category)
    va = subset_for_family(val, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return None

    tok, _ = _load_camembert_once()
    model = copy.deepcopy(_base_model)
    if torch.cuda.is_available():
        model = model.cuda()

    train_ds = _make_dataset(tr, tok, text_field, preprocess)
    val_ds   = _make_dataset(va, tok, text_field, preprocess)
    test_ds  = _make_dataset(te, tok, text_field, preprocess)

    collator = DataCollatorWithPadding(tokenizer=tok)

    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        return {
            "macro_f1":  f1_score(p.label_ids, preds, average="macro",  zero_division=0),
            "accuracy":  accuracy_score(p.label_ids, preds),
        }

    use_fp16 = torch.cuda.is_available()
    out_dir = str(ART / f"cmb_{split_name}_{preprocess}_{text_field}_lr{lr:.0e}")
    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=CAMEMBERT_EPOCHS,
        learning_rate=lr,
        weight_decay=CAMEMBERT_WD,
        warmup_ratio=CAMEMBERT_WARMUP_RATIO,
        lr_scheduler_type="linear",
        per_device_train_batch_size=CAMEMBERT_BATCH,
        per_device_eval_batch_size=CAMEMBERT_BATCH * 2,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=use_fp16,
        label_smoothing_factor=0.1,
        logging_steps=50,
        report_to=[],
        seed=SEED,
        save_total_limit=1,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    # Capture best val macroF1 for sweep comparison
    best_val_f1 = max(
        (log["eval_macro_f1"] for log in trainer.state.log_history if "eval_macro_f1" in log),
        default=0.0,
    )

    pred = trainer.predict(test_ds)
    proba = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()
    y_pred = [LABELS[i] for i in proba.argmax(1)]
    yte = te["outlet"].astype(str).tolist()
    m = eval_metrics(yte, y_pred, proba)

    name = f"camembert|{family}|{category}|{text_field}|{preprocess}|lr{lr:.0e}"
    m.update({"experiment": name, "model": "camembert", "family": family, "category": category,
              "text_field": text_field, "preprocess": preprocess, "split": split_name})
    RESULTS.append(m)
    pc = per_class_df(yte, y_pred, LABELS); pc["experiment"] = name; pc["split"] = split_name
    PER_CLASS.append(pc)
    print(f"  >>> TEST [{name}/{split_name}]  acc={m['accuracy']:.3f}  "
          f"macroF1={m['macro_f1']:.3f}  top2={m['top2_acc']:.3f}  "
          f"(best_val_f1={best_val_f1:.4f})")

    if collect_conf:
        from sklearn.metrics import confusion_matrix as _cm
        cm = _cm(yte, y_pred, labels=LABELS)
        tag = f"{split_name}_camembert"
        CONF_MATRICES[tag] = pd.DataFrame(cm, index=LABELS, columns=LABELS)
        conf = proba.max(axis=1)
        dfp = te[["article_id", "outlet", "category", "year"]].copy()
        dfp["text_len"] = build_text(te, text_field, preprocess).str.len().values
        dfp["pred"] = y_pred
        dfp["confidence"] = conf
        dfp["correct"] = (dfp["outlet"].astype(str) == dfp["pred"])
        dfp["experiment"] = name; dfp["split"] = split_name
        PREDICTIONS.append(dfp)
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_val_f1


## PHASE 1 — Training (seeded, resumable, persisted)

This notebook trains the full model ladder and **persists every run** to Drive so the
analysis notebook (`phase2_analysis.ipynb`) never has to retrain.

What changed vs the prototype:
- **`mask_entities`** added alongside `raw` / `mask_outlet` (the actor/topic confound-killer).
- **Multi-seed** (42 / 13 / 1234) for the neural models on the temporal headline cells.
- **Per-run persistence**: each run writes `runs/preds/<rid>.parquet`,
  `runs/proba/<rid>.npy`, `runs/metrics/<rid>.json` to Drive.
- **Resumable**: a run whose metrics JSON already exists is skipped, so a Colab
  disconnect mid-job loses nothing — just re-run this notebook top-to-bottom.
- CamemBERT re-initialises its classification head **per seed** (true seed variance),
  and Trainer checkpoints go to ephemeral `/content` (Drive stays clean).


In [ ]:
# === Phase-1 persistence + seeding infrastructure =====================
import json, re as _re, numpy as np, pandas as pd

RUNS = ART / "runs"
for sub in ("preds", "proba", "metrics"):
    (RUNS / sub).mkdir(parents=True, exist_ok=True)

SEED_MAIN  = 42
SEEDS_EXTRA = [13, 1234]              # 3 seeds total (42 + 13 + 1234) on temporal headline cells
PREPROCS   = ["raw", "mask_outlet", "mask_entities"]

def _slug(s):
    return _re.sub(r"[^0-9A-Za-z]+", "_", str(s)).strip("_")

def make_rid(model, family, category, tf, pp, split, seed, lr=None):
    base = f"{PANEL_TAG}-{model}-{family}-{category}-{tf}-{pp}-{split}-s{seed}"
    if lr is not None:
        base += f"-lr{lr:.0e}"
    return _slug(base)

def run_done(rid):
    return (RUNS / "metrics" / f"{rid}.json").exists()

def load_meta(rid):
    return json.load(open(RUNS / "metrics" / f"{rid}.json"))

def save_run(rid, meta, dfp, proba=None, proba_labels=None):
    dfp.to_parquet(RUNS / "preds" / f"{rid}.parquet", index=False)
    if proba is not None:
        np.save(RUNS / "proba" / f"{rid}.npy", np.asarray(proba, dtype="float16"))
    meta = dict(meta)
    meta["proba_labels"] = list(proba_labels) if proba_labels is not None else None
    json.dump(meta, open(RUNS / "metrics" / f"{rid}.json", "w"))

def build_preds_df(te_frame, y_true, y_pred, conf, tf, pp, cat_prefix=False):
    dfp = te_frame[["article_id", "category", "year"]].copy()
    dfp["true"] = list(map(str, y_true))
    dfp["pred"] = list(y_pred)
    dfp["confidence"] = np.asarray(conf, dtype="float32")
    dfp["correct"] = (dfp["true"].values == np.asarray(dfp["pred"], dtype=object))
    dfp["text_len"] = build_text(te_frame, tf, pp, cat_prefix).str.len().values
    return dfp

def seed_everything(seed):
    import random as _random
    _random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

print("Persistence ready ->", RUNS)


In [ ]:
# === Majority baseline (persisted) ===================================
def run_majority2(split_name):
    tr, _, te = SPLITS[split_name]
    rid = make_rid("majority", "baseline", "ALL", "headline_only", "none", split_name, 0)
    if run_done(rid):
        print("SKIP", rid); return
    maj = tr["outlet"].astype(str).value_counts().idxmax()
    y_true = te["outlet"].astype(str).tolist()
    y_pred = [maj] * len(y_true)
    m = eval_metrics(y_true, y_pred, None)
    meta = {"rid": rid, "model": "majority", "family": "baseline", "category": "ALL",
            "text_field": "headline_only", "preprocess": "none", "split": split_name,
            "seed": 0, "lr": None, "best_val_f1": None, **m}
    dfp = build_preds_df(te, y_true, y_pred, np.full(len(y_true), np.nan), "headline_only", "raw")
    save_run(rid, meta, dfp)
    print(f"[majority/{split_name}] predict='{maj}' acc={m['accuracy']:.3f}")

for s in SPLITS:
    run_majority2(s)


In [ ]:
# === Linear runner (persisted; collects char/word features on pooled) =
def run_linear2(kind, family, category, text_field, preprocess, split_name,
                seed=SEED_MAIN, cat_prefix=False):
    rid = make_rid(kind + ("_catpfx" if cat_prefix else ""), family, category,
                   text_field, preprocess, split_name, seed)
    if run_done(rid):
        print("SKIP", rid); return load_meta(rid)
    seed_everything(seed)
    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return None
    Xtr = build_text(tr, text_field, preprocess, cat_prefix)
    Xte = build_text(te, text_field, preprocess, cat_prefix)
    ytr = tr["outlet"].astype(str); yte = te["outlet"].astype(str).tolist()

    vec = make_vectorizer(kind); clf = make_clf(kind)
    Xtr_v = vec.fit_transform(Xtr); Xte_v = vec.transform(Xte)
    clf.fit(Xtr_v, ytr)
    y_pred = clf.predict(Xte_v)
    proba_full = None
    if kind == "word_lr":
        proba = clf.predict_proba(Xte_v); proba_labels = list(clf.classes_)
        proba_full = np.zeros((len(yte), len(LABELS)))
        for j, lab in enumerate(proba_labels):
            if lab in LABELS:
                proba_full[:, LABELS.index(lab)] = proba[:, j]
    elif hasattr(clf, "decision_function"):
        # char-SVM (LinearSVC): save softmax(decision_function) as pseudo-proba so the panel can be
        # re-evaluated with an outlet removed WITHOUT refitting (future no-retrain guarantee).
        dfun = clf.decision_function(Xte_v)
        if dfun.ndim == 1:
            dfun = np.column_stack([-dfun, dfun])
        _e = np.exp(dfun - dfun.max(axis=1, keepdims=True)); _sm = _e / _e.sum(axis=1, keepdims=True)
        proba_labels = list(clf.classes_)
        proba_full = np.zeros((len(yte), len(LABELS)))
        for j, lab in enumerate(proba_labels):
            if lab in LABELS:
                proba_full[:, LABELS.index(lab)] = _sm[:, j]
    m = eval_metrics(yte, y_pred, proba_full)
    meta = {"rid": rid, "model": kind + ("_catpfx" if cat_prefix else ""), "family": family,
            "category": category, "text_field": text_field, "preprocess": preprocess,
            "split": split_name, "seed": seed, "lr": None, "best_val_f1": None, **m}
    conf = proba_full.max(axis=1) if proba_full is not None else np.full(len(yte), np.nan)
    dfp = build_preds_df(te, yte, y_pred, conf, text_field, preprocess, cat_prefix)
    save_run(rid, meta, dfp, proba_full, LABELS if proba_full is not None else None)

    # Collect interpretable features on the pooled models (raw + mask_entities).
    if family == "pooled" and category == "ALL" and preprocess in ("raw", "mask_entities") and not cat_prefix:
        if kind == "word_lr":
            extract_word_features(vec, clf, rid)
        else:
            extract_char_features(vec, clf, rid)
    print(f"[{rid}] acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f}")
    return meta


In [ ]:
# === CharCNN runner (persisted, seeded) ==============================
def run_charcnn2(family, category, text_field, preprocess, split_name,
                 model_type="pure", seed=SEED_MAIN, device=None, exp_num=None):
    import torch
    import gc  # Added for memory management
    from sklearn.metrics import f1_score, accuracy_score
    model_key = "charcnn_pure" if model_type == "pure" else "charcnn_word"
    rid = make_rid(model_key, family, category, text_field, preprocess, split_name, seed)

    pfx = f"[{exp_num}] " if exp_num is not None else ""
    if run_done(rid):
        print(f"{pfx}SKIP {rid}"); return load_meta(rid)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_everything(seed)

    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category); va = subset_for_family(val, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return None
    Xtr = build_text(tr, text_field, preprocess).tolist()
    Xva = build_text(va, text_field, preprocess).tolist()
    Xte = build_text(te, text_field, preprocess).tolist()
    ytr = tr["outlet"].astype(str).tolist(); yva = va["outlet"].astype(str).tolist()
    yte = te["outlet"].astype(str).tolist()
    labels = sorted(set(ytr)); lab2id = {l: i for i, l in enumerate(labels)}
    ytr_i = [lab2id[l] for l in ytr]; yva_i = [lab2id[l] for l in yva]; yte_i = [lab2id[l] for l in yte]

    char2idx = build_char_vocab(Xtr); word2idx = build_word_vocab(Xtr)
    g = torch.Generator(); g.manual_seed(seed)

    # --- SURGICAL CHANGE 1: Optimize Data Loaders ---
    train_dl = DataLoader(HeadlineDataset(Xtr, ytr_i, char2idx, word2idx),
                          batch_size=64, shuffle=True, generator=g, # Increased batch size to 64 for CNNs
                          num_workers=2, pin_memory=True)
    val_dl   = DataLoader(HeadlineDataset(Xva, yva_i, char2idx, word2idx), batch_size=128, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(HeadlineDataset(Xte, yte_i, char2idx, word2idx), batch_size=128, num_workers=2, pin_memory=True)
    # ------------------------------------------------

    if model_type == "pure":
        model = CharCNNPureModel(len(char2idx), len(labels)).to(device)
    else:
        model = CharCNNWordModel(len(char2idx), len(word2idx), len(labels)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # --- SURGICAL CHANGE 2: GradScaler for Automatic Mixed Precision (AMP) ---
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    # ------------------------------------------------------------------------

    total_batches = len(train_dl)
    print(f"{pfx}Starting {rid} ({total_batches} batches/epoch)")

    best_f1, best_state, patience = -1, None, 0
    for epoch in range(30):
        model.train()
        for batch_idx, (c, w, y) in enumerate(train_dl, start=1):
            c, w, y = c.to(device, non_blocking=True), w.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()

            # --- SURGICAL CHANGE 3: Run forward pass in FP16 mixed precision ---
            if scaler:
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(c, w), y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(c, w), y)
                loss.backward()
                optimizer.step()
            # ------------------------------------------------------------------

            if batch_idx % 5 == 0 or batch_idx == total_batches:
                print(f"\r   → Epoch {epoch+1:02d}/30 | Batch {batch_idx:04d}/{total_batches:04d} | Loss: {loss.item():.4f}", end="", flush=True)

        model.eval(); preds, truths = [], []
        with torch.no_grad():
            for c, w, y in val_dl:
                c, w = c.to(device, non_blocking=True), w.to(device, non_blocking=True)
                if scaler:
                    with torch.amp.autocast('cuda'):
                        out = model(c, w)
                else:
                    out = model(c, w)
                preds.extend(out.argmax(1).cpu().numpy()); truths.extend(y.numpy())
        val_f1 = f1_score(truths, preds, average="macro", zero_division=0)

        print(f" | Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            # --- SURGICAL CHANGE 4: Clone directly on GPU device to save time ---
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= 3:
                print(f"   Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_state); model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for c, w, _ in test_dl:
            c, w = c.to(device, non_blocking=True), w.to(device, non_blocking=True)
            if scaler:
                with torch.amp.autocast('cuda'):
                    out = model(c, w)
            else:
                out = model(c, w)
            all_probs.extend(F.softmax(out, 1).cpu().numpy()); all_preds.extend(out.argmax(1).cpu().numpy())
    proba = np.array(all_probs)
    proba_full = np.zeros((len(yte), len(LABELS)))
    for j, lab in enumerate(labels):
        if lab in LABELS:
            proba_full[:, LABELS.index(lab)] = proba[:, j]
    y_pred = [labels[i] for i in all_preds]
    m = eval_metrics(yte, y_pred, proba_full)
    meta = {"rid": rid, "model": model_key, "family": family, "category": category,
            "text_field": text_field, "preprocess": preprocess, "split": split_name,
            "seed": seed, "lr": None, "best_val_f1": float(best_f1), **m}
    dfp = build_preds_df(te, yte, y_pred, proba_full.max(1), text_field, preprocess)
    save_run(rid, meta, dfp, proba_full, LABELS)

    # Clean up forcefully
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"{pfx}[{rid}] acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f} top2={m['top2_acc']:.3f}")
    return meta

In [ ]:
# === CamemBERT runner (persisted, per-seed fresh head; L4-optimized) =====
def _new_camembert(seed):
    from transformers import AutoModelForSequenceClassification, set_seed
    import torch
    set_seed(seed)  # makes the freshly-initialised classification head seed-dependent
    model = AutoModelForSequenceClassification.from_pretrained(
        CAMEMBERT_MODEL, num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id={l: i for i, l in enumerate(LABELS)})
    return model.cuda() if torch.cuda.is_available() else model

def run_camembert2(family, category, text_field, preprocess, split_name, seed=SEED_MAIN, lr=None, exp_num=None):
    import torch, gc
    from sklearn.metrics import f1_score, accuracy_score
    # Ada/L4: enable TF32 matmuls (speeds fp32 paths; bf16 used for the model).
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    lr = lr if lr is not None else CAMEMBERT_LR
    rid = make_rid("camembert", family, category, text_field, preprocess, split_name, seed, lr)
    pfx = f"[{exp_num}] " if exp_num is not None else ""
    if run_done(rid):
        meta = load_meta(rid); print(f"{pfx}SKIP {rid}"); return meta.get("best_val_f1")
    tok, _ = _load_camembert_once()
    seed_everything(seed)
    model = _new_camembert(seed)

    tr, val, te = SPLITS[split_name]
    tr = subset_for_family(tr, family, category); va = subset_for_family(val, family, category)
    te = subset_for_family(te, family, category)
    if len(tr) < 50 or len(te) < 10:
        return None
    train_ds = _make_dataset(tr, tok, text_field, preprocess)
    val_ds   = _make_dataset(va, tok, text_field, preprocess)
    test_ds  = _make_dataset(te, tok, text_field, preprocess)
    # pad_to_multiple_of=8 keeps tensor cores fed.
    collator = DataCollatorWithPadding(tokenizer=tok, pad_to_multiple_of=8)

    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        return {"macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0),
                "accuracy": accuracy_score(p.label_ids, preds)}

    use_cuda = torch.cuda.is_available()
    args = TrainingArguments(
        output_dir=f"/content/cmb_tmp/{rid}",      # ephemeral; keep Drive clean
        num_train_epochs=CAMEMBERT_EPOCHS, learning_rate=lr, weight_decay=CAMEMBERT_WD,
        warmup_ratio=CAMEMBERT_WARMUP_RATIO, lr_scheduler_type="linear",
        per_device_train_batch_size=CAMEMBERT_BATCH,         # L4 24GB fits 128 at seq<=64; lower to 64 if OOM
        per_device_eval_batch_size=CAMEMBERT_BATCH,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True,
        bf16=use_cuda, fp16=False, tf32=use_cuda,           # bf16 compute + tf32 matmuls on Ada
        optim="adamw_torch_fused",                          # fused optimizer step (faster); -> "adamw_torch" if unsupported
        dataloader_num_workers=8, dataloader_pin_memory=True,
        dataloader_persistent_workers=True, dataloader_prefetch_factor=4,
        label_smoothing_factor=0.1, logging_steps=50,
        report_to=[], seed=seed, data_seed=seed, save_total_limit=1)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collator, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    print(f"{pfx}Starting {rid} (train_bs={CAMEMBERT_BATCH})")
    trainer.train()
    best_val_f1 = max((lg["eval_macro_f1"] for lg in trainer.state.log_history if "eval_macro_f1" in lg), default=0.0)
    pred = trainer.predict(test_ds)
    proba = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()
    y_pred = [LABELS[i] for i in proba.argmax(1)]
    yte = te["outlet"].astype(str).tolist()
    m = eval_metrics(yte, y_pred, proba)
    meta = {"rid": rid, "model": "camembert", "family": family, "category": category,
            "text_field": text_field, "preprocess": preprocess, "split": split_name,
            "seed": seed, "lr": float(lr), "best_val_f1": float(best_val_f1), **m}
    dfp = build_preds_df(te, yte, y_pred, proba.max(1), text_field, preprocess)
    save_run(rid, meta, dfp, proba, LABELS)
    del model, trainer
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    print(f"{pfx}[{rid}] acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f} top2={m['top2_acc']:.3f} val={best_val_f1:.4f}")
    return best_val_f1


In [ ]:
# === PREFLIGHT: validate training prerequisites BEFORE the expensive run ===
# Fail fast on missing data, broken deps, or a feature-extraction path that would
# silently produce empty outputs -- so we never burn GPU hours on a broken setup.
def _phase1_preflight():
    problems = []
    need = {"outlet", "title", "category", "year"}
    assert df_sample is not None and not df_sample.empty, "df_sample is empty/missing"
    miss = need - set(df_sample.columns)
    if miss: problems.append(f"df_sample missing columns: {miss}")
    if df_sample["outlet"].nunique() != len(LABELS):
        problems.append(f"df_sample has {df_sample['outlet'].nunique()} outlets, expected {len(LABELS)}")
    cell = df_sample.groupby(["outlet", "category"]).size()
    if cell.min() < 100:
        problems.append(f"degenerate outlet x category cell (min={cell.min()})")
    print(f"sample: {len(df_sample):,} rows | {df_sample['outlet'].nunique()} outlets "
          f"| cell min/max = {cell.min()}/{cell.max()}")
    # splits non-empty + temporal test is the held-out year
    assert len(SPLITS["temporal"][2]) > 0 and len(SPLITS["random"][2]) > 0, "empty test split"
    print(f"splits: random-test={len(SPLITS['random'][2]):,} temporal-test={len(SPLITS['temporal'][2]):,}")
    # spaCy NER dependency (mask_entities) actually loads and runs
    _t = mask_entities("Le president rencontre Emmanuel Macron a Paris.")
    print("mask_entities smoke:", _t)
    # linear fit + coefficient extraction smoke (the path that produced the empty parquet)
    _s = df_sample.sample(min(300, len(df_sample)), random_state=0)
    for _k in ("char_svm", "word_lr"):
        _v = make_vectorizer(_k); _c = make_clf(_k)
        _c.fit(_v.fit_transform(build_text(_s, "headline_only", "raw")), _s["outlet"].astype(str))
        assert hasattr(_c, "coef_") and _c.coef_.shape[1] == len(_v.get_feature_names_out()), \
            f"{_k}: coefficient extraction broken"
    print("linear + feature-extraction smoke: OK")
    # filesystem / parquet engine writable
    _tp = ART / "_preflight_write_test.parquet"
    pd.DataFrame({"x": [1]}).to_parquet(_tp); _tp.unlink()
    print("artifact write: OK")
    if problems:
        raise AssertionError("PHASE-1 PREFLIGHT FAILED: " + "; ".join(problems))
    print(f"PHASE-1 PREFLIGHT OK (panel={PANEL_TAG}) -- safe to train.")

_phase1_preflight()


In [ ]:
# === EXECUTION DRIVER (resumable; safe to re-run after disconnect) ====
# Linear + majority: deterministic, single seed (SEED_MAIN).
for split in SPLITS:
    for fam, cats in [("within", CATEGORIES), ("pooled", ["ALL"])]:
        for cat in cats:
            for pp in PREPROCS:
                for kind in ["word_lr", "char_svm"]:
                    run_linear2(kind, fam, cat, "headline_only", pp, split, seed=SEED_MAIN)
    run_linear2("word_lr", "pooled", "ALL", "headline_only", "raw", split, seed=SEED_MAIN, cat_prefix=True)
print("\nLinear matrix done.")

# CharCNN: all cells at SEED_MAIN; temporal cells also get the extra seeds.
import torch
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", dev)

# Pre-calculate the exact total of valid iterations to display clean bounds
total_experiments = 0
for split in SPLITS:
    for pp in PREPROCS:
        for arch in ["pure", "word"]:
            seeds = [SEED_MAIN] + (SEEDS_EXTRA if split == "temporal" else [])
            total_experiments += len(seeds)

print(f"Total experiments planned in matrix: {total_experiments}\n")

exp_counter = 1
for split in SPLITS:
    for pp in PREPROCS:
        for arch in ["pure", "word"]:
            seeds = [SEED_MAIN] + (SEEDS_EXTRA if split == "temporal" else [])
            for sd in seeds:
                # Build the dynamic progress index string (e.g., "1/48")
                exp_progress_str = f"{exp_counter}/{total_experiments}"

                run_charcnn2(
                    "pooled", "ALL", "headline_only", pp, split,
                    model_type=arch, seed=sd, device=dev,
                    exp_num=exp_progress_str
                )
                exp_counter += 1

print("\nCharCNN matrix done.")

# CamemBERT: LR sweep (raw/random) -> best LR -> full grid -> extra temporal seeds.
if RUN_CAMEMBERT:
    import json
    sweep_path = RUNS / "camembert_lr_sweep.json"

    # Resolve best_lr: load from sweep file if exists, else run sweep now.
    if sweep_path.exists():
        print(f"\n== Found existing sweep file: {sweep_path} — loading best_lr ==")
        data = json.load(open(sweep_path, "r"))
        sweep = data["sweep"]
        best_lr = data["best_lr"]
        total_sweep = 0
    else:
        print("\n== No sweep file — running LR sweep (headline_only/raw/random) ==")
        total_sweep = len(CAMEMBERT_LR_SWEEP)

    # 1. Pre-calculate total experiments
    # raw/random always runs exactly once (either in sweep or as explicit step below)
    total_grid = 0
    for split in SPLITS:
        for pp in PREPROCS:
            if split == "random" and pp == "raw":
                continue
            total_grid += 1

    total_extra_seeds = len(PREPROCS) * len(SEEDS_EXTRA)
    total_experiments = total_sweep + 1 + total_grid + total_extra_seeds  # +1 = raw/random at best_lr

    print(f"Total CamemBERT experiments planned: {total_experiments}\n")

    exp_counter = 1

    # 2. LR sweep (skipped if sweep file already existed)
    if not sweep_path.exists():
        sweep = {}
        for lr in CAMEMBERT_LR_SWEEP:
            exp_progress_str = f"{exp_counter}/{total_experiments}"
            v = run_camembert2("pooled", "ALL", "headline_only", "raw", "random",
                              seed=SEED_MAIN, lr=lr, exp_num=exp_progress_str)
            sweep[f"{lr:.0e}"] = v if v is not None else 0.0
            exp_counter += 1
        best_lr = max(CAMEMBERT_LR_SWEEP, key=lambda lr: sweep.get(f"{lr:.0e}", 0) or 0)
        json.dump({"sweep": sweep, "best_lr": best_lr}, open(sweep_path, "w"))
    # -------------------------------------------------------------------------

    print("Best LR:", f"{best_lr:.0e}", "| sweep:", sweep)

    # 3. Always ensure raw/random at best_lr is in this run's metrics folder.
    #    run_camembert2 skips internally if the RID already exists (resumable).
    exp_progress_str = f"{exp_counter}/{total_experiments}"
    run_camembert2("pooled", "ALL", "headline_only", "raw", "random",
                   seed=SEED_MAIN, lr=best_lr, exp_num=exp_progress_str)
    exp_counter += 1

    # 2. Main Grid Execution
    for split in SPLITS:
        for pp in PREPROCS:
            if split == "random" and pp == "raw":
                continue   # already produced in the sweep at best_lr
            exp_progress_str = f"{exp_counter}/{total_experiments}"
            run_camembert2("pooled", "ALL", "headline_only", pp, split,
                              seed=SEED_MAIN, lr=best_lr, exp_num=exp_progress_str)
            exp_counter += 1

    # 3. Extra Seeds Execution
    for pp in PREPROCS:
        for sd in SEEDS_EXTRA:
            exp_progress_str = f"{exp_counter}/{total_experiments}"
            run_camembert2("pooled", "ALL", "headline_only", pp, "temporal",
                              seed=sd, lr=best_lr, exp_num=exp_progress_str)
            exp_counter += 1

    print("\nCamemBERT matrix done.")
else:
    print("RUN_CAMEMBERT=False — skipping transformer.")

In [ ]:
# === Save feature importances, manifest, and the sampled dataset ======
# Recompute linear features deterministically from the sample so the parquet is
# ALWAYS complete, even on resumed runs where the linear cells were skipped
# (in-run FEATURE_IMP would otherwise be empty and overwrite the file).
_rows = []
_trf = SPLITS["random"][0]
_ytr = _trf["outlet"].astype(str)
for _kind in ("char_svm", "word_lr"):
    for _pp in ("raw", "mask_entities"):
        _vec = make_vectorizer(_kind); _clf = make_clf(_kind)
        _clf.fit(_vec.fit_transform(build_text(_trf, "headline_only", _pp)), _ytr)
        _feats = np.array(_vec.get_feature_names_out())
        for _i, _lab in enumerate(_clf.classes_):
            _coef = _clf.coef_[_i] if _clf.coef_.shape[0] > 1 else _clf.coef_[0]
            for _rank, _j in enumerate(np.argsort(_coef)[-20:][::-1]):
                _rows.append({"experiment": f"{_kind}|pooled|ALL|headline_only|{_pp}",
                              "model": _kind, "preprocess": _pp, "outlet": _lab,
                              "rank": _rank + 1,
                              "feature": repr(_feats[_j]) if _kind == "char_svm" else _feats[_j],
                              "weight": float(_coef[_j])})
feat = pd.DataFrame(_rows)
feat.to_parquet(ART / "feature_importance.parquet", index=False)
print("feature rows:", len(feat))

metas = [json.load(open(p)) for p in sorted((RUNS / "metrics").glob("*.json"))]
manifest = pd.DataFrame(metas)
keep = ["rid","model","family","category","text_field","preprocess","split","seed","lr",
        "accuracy","macro_f1","weighted_f1","top2_acc","n","best_val_f1"]
manifest = manifest[[c for c in keep if c in manifest.columns]].sort_values(
    ["split","model","preprocess","seed"])
manifest.to_csv(ART / "run_manifest.csv", index=False)
df_sample.to_parquet(ART / "sampled_dataset.parquet", index=False)
df_sample.to_parquet(ART / f"sampled_dataset_{PANEL_TAG}.parquet", index=False)  # panel-tagged copy for Phase 2

print("\n=== RUN MANIFEST (", len(manifest), "runs ) ===")
print(manifest.to_string(index=False))
print("\nPHASE 1 COMPLETE. All runs persisted under:", RUNS)
print("Now run phase2_analysis.ipynb (CPU only).")
